# Radar de Concursos — Administração

Edite órgãos, concursos, cargos, vacância e alertas. Ao final, teste, visualize e exporte um ZIP limpo.

In [ ]:
import os, pathlib, subprocess, sys
REPO_URL="https://github.com/osmarrcs/radar-concursos-ti.git"
ROOT=pathlib.Path("/content/radar-concursos-ti")
if not ROOT.exists(): subprocess.run(["git","clone","--depth","1",REPO_URL,str(ROOT)],check=True)
os.chdir(ROOT)
sys.path.insert(0,str(ROOT/"src"))
print("Projeto carregado em",ROOT)

In [ ]:
from pathlib import Path
from radar_concursos.repository import load_dataset
from radar_concursos.services import save_organ, save_contest, save_position, update_vacancy, add_alert_source, set_alert_selection
from IPython.display import display
import ipywidgets as widgets

def refresh():
    global DATA
    DATA=load_dataset()
    return DATA
refresh()
print(f"{len(DATA['organs']['organs'])} órgãos, {len(DATA['contests']['contests'])} concursos, {len(DATA['positions']['positions'])} cargos")

## 1. Cadastrar órgão
Somente nome e sigla são obrigatórios. Os demais campos são inferidos e podem ser corrigidos depois no JSON.

In [ ]:
org_name=widgets.Text(description="Nome",layout=widgets.Layout(width="90%"))
org_acronym=widgets.Text(description="Sigla")
org_button=widgets.Button(description="Salvar órgão",button_style="success")
org_output=widgets.Output()
def on_org(_):
    with org_output:
        org_output.clear_output()
        try:
            print(save_organ(org_name.value,org_acronym.value))
            refresh()
        except Exception as exc: print("ERRO:",exc)
org_button.on_click(on_org)
display(org_name,org_acronym,org_button,org_output)

## 1.1 Cadastrar fonte oficial para alertas
Informe a página de notícias, concursos ou um feed RSS do órgão.

In [ ]:
refresh()
s_org=widgets.Dropdown(options=[(f"{x['acronym']} — {x['name']}",x['id']) for x in DATA['organs']['organs']],description="Órgão",layout=widgets.Layout(width="90%"))
s_label=widgets.Text(description="Rótulo",placeholder="Página oficial de concursos",layout=widgets.Layout(width="90%"))
s_url=widgets.Text(description="URL",placeholder="https://...",layout=widgets.Layout(width="95%"))
s_type=widgets.Dropdown(options=[("Página HTML","html"),("RSS","rss"),("Atom","atom")],description="Tipo")
s_button=widgets.Button(description="Salvar fonte",button_style="success")
s_output=widgets.Output()
def on_source(_):
    with s_output:
        s_output.clear_output()
        try:
            print(add_alert_source(s_org.value,s_label.value,s_url.value,s_type.value)); refresh()
        except Exception as exc: print("ERRO:",exc)
s_button.on_click(on_source)
display(s_org,s_label,s_url,s_type,s_button,s_output)

## 2. Cadastrar concurso
Cadastre o edital uma única vez. As fontes e a validade ficarão compartilhadas por todos os cargos.

In [ ]:
refresh()
org_options=[(f"{x['acronym']} — {x['name']}",x['id']) for x in DATA['organs']['organs']]
c_org=widgets.Dropdown(options=org_options,description="Órgão")
c_title=widgets.Text(description="Título",layout=widgets.Layout(width="90%"))
c_year=widgets.IntText(description="Ano",value=2026)
c_status=widgets.Text(description="Status",value="Previsto")
c_valid=widgets.Text(description="Validade",placeholder="YYYY-MM-DD")
c_official=widgets.Checkbox(description="Dados oficiais",value=False)
c_button=widgets.Button(description="Salvar concurso",button_style="success")
c_output=widgets.Output()
def on_contest(_):
    with c_output:
        c_output.clear_output()
        try:
            item=save_contest({"organ_id":c_org.value,"title":c_title.value,"year":c_year.value,"status":c_status.value,"valid_until":c_valid.value,"reserve_list":None,"lotation":"","confidence":"","is_official":c_official.value,"sources":[],"collection_status":{},"notes":""})
            print(item); refresh()
        except Exception as exc: print("ERRO:",exc)
c_button.on_click(on_contest)
display(c_org,c_title,c_year,c_status,c_valid,c_official,c_button,c_output)

## 3. Cadastrar cargo ou especialidade
O sistema aceita qualquer nomenclatura do edital e não filtra por área.

In [ ]:
refresh()
contest_map={x['id']:x for x in DATA['contests']['contests']}
organ_map={x['id']:x for x in DATA['organs']['organs']}
p_options=[(f"{organ_map[x['organ_id']]['acronym']} — {x['year']} — {x['title']}",x['id']) for x in DATA['contests']['contests']]
p_contest=widgets.Dropdown(options=p_options,description="Concurso",layout=widgets.Layout(width="90%"))
p_position=widgets.Text(description="Cargo",layout=widgets.Layout(width="90%"))
p_specialty=widgets.Text(description="Especialidade",layout=widgets.Layout(width="90%"))
p_code=widgets.Text(description="Código")
p_vacancies=widgets.IntText(description="Vagas",value=0)
p_quota=widgets.Text(description="Modalidade",value="Ampla concorrência")
p_button=widgets.Button(description="Salvar cargo",button_style="success")
p_output=widgets.Output()
def on_position(_):
    with p_output:
        p_output.clear_output()
        try:
            item=save_position({"contest_id":p_contest.value,"position":p_position.value,"specialty":p_specialty.value,"position_code":p_code.value,"immediate_vacancies":p_vacancies.value,"last_called_rank":None,"last_called_score":None,"total_appointed":None,"quota_type":p_quota.value,"sources":[],"collection_status":{},"notes":""})
            print(item); refresh()
        except Exception as exc: print("ERRO:",exc)
p_button.on_click(on_position)
display(p_contest,p_position,p_specialty,p_code,p_vacancies,p_quota,p_button,p_output)

## 4. Atualizar vacância separadamente

In [ ]:
refresh()
pos_options=[(f"{x['id']} — {x['position']} — {x.get('specialty','')}",x['id']) for x in DATA['positions']['positions']]
v_position=widgets.Dropdown(options=pos_options,description="Cargo",layout=widgets.Layout(width="95%"))
v_count=widgets.IntText(description="Vagos",value=0)
v_date=widgets.Text(description="Data",placeholder="YYYY-MM-DD")
v_reason=widgets.Textarea(description="Observação",layout=widgets.Layout(width="95%"))
v_button=widgets.Button(description="Salvar vacância",button_style="success")
v_output=widgets.Output()
def on_vacancy(_):
    with v_output:
        v_output.clear_output()
        try:
            item=update_vacancy(v_position.value,{"count":v_count.value,"reference_date":v_date.value,"reason":v_reason.value or "Vacância informada pelo usuário.","sources":[],"collection_status":{"found":False,"code":"MISSING_OFFICIAL_SOURCE","reason":"Inclua a fonte oficial da vacância no JSON.","source":"vacancia"}})
            print(item); refresh()
        except Exception as exc: print("ERRO:",exc)
v_button.on_click(on_vacancy)
display(v_position,v_count,v_date,v_reason,v_button,v_output)

## 5. Selecionar órgãos para alertas

In [ ]:
refresh()
a_options=[(f"{x['acronym']} — {x['name']}",x['id']) for x in DATA['organs']['organs']]
a_select=widgets.SelectMultiple(options=a_options,description="Órgãos",layout=widgets.Layout(width="95%",height="180px"))
a_select.value=tuple(DATA['alerts'].get('monitored_organs',[]))
a_button=widgets.Button(description="Salvar alertas",button_style="success")
a_output=widgets.Output()
def on_alerts(_):
    with a_output:
        a_output.clear_output()
        try: print(set_alert_selection(list(a_select.value))); refresh()
        except Exception as exc: print("ERRO:",exc)
a_button.on_click(on_alerts)
display(a_select,a_button,a_output)

## 6. Testar e gerar o portal

In [ ]:
import subprocess, sys
subprocess.run([sys.executable,"-m","unittest","discover","-s","tests","-v"],check=True)
subprocess.run([sys.executable,"-m","radar_concursos.build"],check=True)

## 7. Visualizar

In [ ]:
import threading, http.server, socketserver
from google.colab import output
PORT=8000
os.chdir(ROOT/"dist")
class ReuseTCPServer(socketserver.TCPServer): allow_reuse_address=True
try:
    server.shutdown()
except Exception: pass
server=ReuseTCPServer(("",PORT),http.server.SimpleHTTPRequestHandler)
threading.Thread(target=server.serve_forever,daemon=True).start()
output.serve_kernel_port_as_iframe(PORT,height=900)
os.chdir(ROOT)

## 8. Exportar ZIP limpo

In [ ]:
import shutil, tempfile, pathlib
from google.colab import files
export=pathlib.Path("/content/radar-concursos-ti-atualizado.zip")
with tempfile.TemporaryDirectory() as tmp:
    target=pathlib.Path(tmp)/"radar-concursos-ti"
    shutil.copytree(ROOT,target,ignore=shutil.ignore_patterns(".git","__pycache__","*.pyc",".pytest_cache",".DS_Store","dist","alert_last_run.json"))
    shutil.make_archive(str(export.with_suffix('')),"zip",root_dir=target)
files.download(str(export))